<a href="https://colab.research.google.com/github/jonhrh1348/aviation-delay/blob/feature%2Fapi_data_setup/00_Initial_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark
!pip install kafka-python
!pip install clickhouse-connect


# kafka-python>=2.3.0
# pyspark>=4.0.0
# clickhouse-connect>=0.15.0
#


In [ ]:
import clickhouse_connect
import datetime
import json
import requests
import time
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient, TopicPartition
from google.colab import userdata
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import SparkContext, SparkConf

In [ ]:
# 1a. Start the SparkSession builder or continue previous one
#  master("local")

spark = SparkSession.builder \
    .appName("PySpark-InitialSetup") \
    .config("spark.driver.memory", "12g") \
    .getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

print("Spark Session created successfully!")

In [ ]:
# 1b. Start the Clickhouse Client for db use
ch_user = userdata.get('CLICKHOUSE_USER')
ch_password = userdata.get('CLICKHOUSE_PW')

client = clickhouse_connect.get_client(
    host='l21wgze3j3.asia-southeast1.gcp.clickhouse.cloud',
    user=ch_user,
    password=ch_password,
    secure=True
)
print(f'Client connected')

In [ ]:
### KAFKA server setup with bash

# 1. Download Kafka binaries
!curl -sSOL https://archive.apache.org/dist/kafka/3.5.1/kafka_2.13-3.5.1.tgz
!tar -xzf kafka_2.13-3.5.1.tgz

# 2. Start Zookeeper (background)
!nohup kafka_2.13-3.5.1/bin/zookeeper-server-start.sh kafka_2.13-3.5.1/config/zookeeper.properties > zookeeper.log 2>&1 &

# 3. Start Kafka Broker (background)
!nohup kafka_2.13-3.5.1/bin/kafka-server-start.sh kafka_2.13-3.5.1/config/server.properties > kafka.log 2>&1 &

# 4. Wait for Kafka to initialize
import time
time.sleep(15)

print("Kafka and Zookeeper started in the background.")

In [ ]:
# Configuration for Kafka
BOOTSTRAP_SERVERS = ['localhost:9092']
TOPIC_NAME = 'raw_aviation_data'  # 'raw_weather_data'

# 1. Kafka Producer
def create_kafka_producer(client_id, bootstrap_servers):
  return KafkaProducer(
    bootstrap_servers=bootstrap_servers,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    client_id=client_id
  )

# 2. Kafka Consumer
def create_kafka_consumer(group_id, bootstrap_servers, topic_name):
  return KafkaConsumer(
      topic_name,
      bootstrap_servers=bootstrap_servers,
      group_id=group_id,
      value_deserializer=lambda x: json.loads(x.decode('utf-8')),
      consumer_timeout_ms=10000
  )

# Streaming function
def stream_to_kafka(data_list, producer, topic_name): # topic name should match that given by consumer
    print(f"Starting data streaming to Kafka with producer client_id: {producer.config['client_id']}")
    for record in data_list:
        producer.send(topic_name, record)
    producer.flush()
    print(f"Sent {len(data_list)} records to Kafka topic: {topic_name}")


def insert_to_clickhouse(consumer_instance, table_name, column_names, defaults):
    print(f"Starting consumption of data from Kafka with consumer_instance: {consumer_instance.config['group_id']}")
    batch= []
    for message in consumer_instance:
      row_item = message.value
      # process raw data logic

      match table_name:
        case 'raw_aviation_flights':
          row = [row_item.get(col, defaults[col]) for col in column_names[:6]]
          cs_item = row_item.get('codeshared', {})
          row.append(cs_item.get('airline', {}))
          row.append(cs_item.get('flight', {}))

        case 'historical_weather_data':
          weather_time = row_item.get('dt', 0)
          row = [row_item.get(col, defaults[col]) for col in column_names[1:]]
          row.insert(0, weather_time)

        case _:
          print (f'{table_name} is not found in database')
          continue

      batch.append(row)

      # Batch insert every 500 records
      if len(batch) >= 500:
          client.insert(table_name, batch, column_names=column_names)
          print(f"Successfully inserted {len(batch)} records to ClickHouse.")
          batch = []

    # Final insert for remaining records
    if batch:
        client.insert(table_name, batch, column_names=column_names)
        print(f"Successfully inserted {len(batch)} remaining records to ClickHouse.")

In [ ]:
# Drop table for raw_aviation_flights if required
client.command(f'''
  DROP TABLE IF EXISTS raw_aviation_flights SYNC
''')

# Create table raw_aviation_flights with JSON types for flexible nested structures
client.command('''
CREATE TABLE IF NOT EXISTS raw_aviation_flights (
    id UUID DEFAULT generateUUIDv4(),
    insert_time DateTime64(3, 'Asia/Singapore') DEFAULT now64(),
    type String,
    status String,
    departure JSON,
    arrival JSON,
    airline Map(String, String),
    flight Map(String, String),
    codeshared_airline Map(String, String),
    codeshared_flight Map(String, String)
) ENGINE = MergeTree()
PRIMARY KEY (id);
''')

print(f"Table raw_aviation_flights verified/created with UUID and Singapore timezone.")

In [ ]:
# Drop table for historical_weather_data if required
client.command(f'''
  DROP TABLE IF EXISTS historical_weather_data SYNC
''')

# Create table historical_weather_data with JSON types for flexible nested structures
client.command('''
CREATE TABLE IF NOT EXISTS historical_weather_data (
    id UUID DEFAULT generateUUIDv4(),
    insert_time DateTime64(3, 'Asia/Singapore') DEFAULT now64(),
    weather_time DateTime,
    main JSON,
    wind JSON,
    clouds JSON,
    rain JSON,
    snow JSON,
    weather Array(JSON)
) ENGINE = MergeTree()
PRIMARY KEY (id);
''')

print(f"Table historical_weather_data verified/created with UUID and Singapore timezone.")

In [ ]:
# 2. Get aviation data from Aviation Edge API
ae_api_key = userdata.get('AE_API_KEY')
code = 'SIN'
flight_types = ['departure', 'arrival']
all_flight_data = []
column_names = ['type', 'status', 'departure', 'arrival',
  'airline', 'flight', 'codeshared_airline', 'codeshared_flight']
defaults = {
  'type': '',
  'status': '',
  'departure': {},
  'arrival': {},
  'airline': {},
  'flight': {},
  'codeshared_airline': {},
  'codeshared_flight': {}
}

hist_av_prod = create_kafka_producer('historical_aviation_data', BOOTSTRAP_SERVERS)
print('hist_av_prod success')
hist_av_cons = create_kafka_consumer('consume_aviation_data', BOOTSTRAP_SERVERS, 'raw_aviation_data')
print('hist_av_cons success')

start_date = datetime.datetime(2026, 4, 2)
end_date = datetime.datetime(2026, 4, 5)  # it must be today-3 or older
interval = datetime.timedelta(days=7)

aviation_api_url = "https://aviation-edge.com/v2/public/flightsHistory"

print(f"Starting data collection and storage from {start_date.date()} to {end_date.date()}.")

for f_type in flight_types:
    current_date = start_date
    while current_date < end_date:
        date_end = min(current_date + interval, end_date)
        date_from = current_date.strftime("%Y-%m-%d")
        date_to = date_end.strftime("%Y-%m-%d")

        aviation_request_url = (
            f'{aviation_api_url}?key={ae_api_key}&code={code}'
            f'&type={f_type}&date_from={date_from}&date_to={date_to}'
        )

        try:
            print(f"Fetching and Saving {f_type} for: {date_from} to {date_to}")
            aviation_response = requests.get(aviation_request_url)
            aviation_data = aviation_response.json()

            if isinstance(aviation_data, list):
            # give the data to kafka topic to process
              stream_to_kafka(aviation_data, hist_av_prod, 'raw_aviation_data')
            # insert data to clickhouse for storage
              insert_to_clickhouse(hist_av_cons, 'raw_aviation_flights', column_names, defaults)

            all_flight_data.extend(aviation_data)

        except Exception as e:
            print(f"An error occurred during API fetch or DB insert: {e}")

        current_date = date_end

print(f"Data collection and DB storage complete. Total records collected: {len(all_flight_data)}")

In [ ]:
# for flight in data:
#     rows_to_insert.append([
#         flight.get('flight', {}).get('iataNumber'),
#         flight.get('airline', {}).get('name'),
#         flight.get('arrival', {}).get('scheduledTime'),
#         flight.get('arrival', {}).get('actualTime'),
#         flight.get('arrival', {}).get('delay'),
#         flight.get('arrival', {}).get('baggage'),
#         flight.get('departure', {}).get('scheduledTime'),
#         flight.get('departure', {}).get('actualTime'),
#         flight.get('departure', {}).get('delay'),
#         flight.get('codeshared', {}).get('airline', {}).get('name'),
#         flight.get('codeshared', {}).get('flight', {}).get('iataNumber')
#     ])

## Insert batch into ClickHouse
# client.insert('aviation_flights', rows_to_insert, column_names=[
#     'IATA_number', 'Airline', 'Arr_scheduled_time', 'Arr_actual_time', 'Arr_delay', 'Arr_baggage',
#     'Dep_scheduled_time', 'Dep_actual_time', 'Dep_delay', 'Codeshared_airline', 'Codeshared_flight_number'
# ])

In [ ]:
# print(json.dumps(all_flight_data[0], indent=2))

# 4. Parallelize the JSON string into a flight collection RDD
flight_collection = spark.sparkContext.parallelize(all_flight_data, 10)

# 5. Create a DataFrame for analysis
aviation_df = (spark.read.json(flight_collection)
    .select(
        "flight.iataNumber",
        "airline.name",
        "arrival.scheduledTime",
        "arrival.actualTime",
        "arrival.delay",
        "arrival.baggage",
        "departure.scheduledTime",
        "departure.actualTime",
        "departure.delay",
        "codeshared.airline.name",
        "codeshared.flight.iataNumber",)
    .toDF(
        "IATA_number",
        "Airline",
        "Arr_scheduled_time",
        "Arr_actual_time",
        "Arr_delay (mins)",
        "Arr_baggage (mins)",
        "Dep_scheduled_time",
        "Dep_actual_time",
        "Dep_delay (mins)",
        "Codeshared_airline",
        "Codeshared_flight_number")
)

# df.printSchema()
aviation_df.show(n=3, truncate=False)

In [ ]:
#
# New set of calls
# 1. Get historical weather data from Open Weather API (to handle batch, db storage and multiple cities later)

lat = 1.290
long = 103.852
ttl_cnt = 0
units_used = 'metric'
weather_api_key = userdata.get('WEATHER_API_KEY')
weather_api_url = "https://history.openweathermap.org/data/2.5/history/city"

# Define the start and end dates
current_date = datetime.datetime(2026, 4, 1)
end_date = datetime.datetime(2026, 4, 8)
interval = datetime.timedelta(days=7)
all_weather_data = []
column_names = ['weather_time', 'main', 'wind', 'clouds', 'rain', 'snow', 'weather']
defaults = {
  'weather_time': 0,
  'main': {},
  'wind': {},
  'clouds': {},
  'rain': {},
  'snow': {},
  'weather': []
}

hist_we_prod = create_kafka_producer('historical_weather_data', BOOTSTRAP_SERVERS)
print('hist_we_prod success')
hist_we_cons = create_kafka_consumer('consume_weather_data', BOOTSTRAP_SERVERS, 'raw_weather_data')
print('hist_we_cons success')

print(f"Iterating through Mar & Apr 2026 and collecting data:\n")

# 2. Batch call API for weather data
while current_date < end_date:
    segment_end = min(current_date + interval, end_date)
    start_epoch = int(current_date.timestamp())
    end_epoch = int(segment_end.timestamp())

    # Construct the URL for this specific window
    weather_request_url = (
        f'{weather_api_url}?lat={lat}&lon={long}'
        f'&type=hour&start={start_epoch}&end={end_epoch}'
        f'&appid={weather_api_key}&units={units_used}'
    )
    if ttl_cnt > 0:
      weather_request_url = f'{weather_request_url}&cnt={ttl_cnt}'

    try:
      print(f"Fetching Period: {current_date.date()} to {segment_end.date()}")
      weather_response = requests.get(weather_request_url)

      resp_data = weather_response.json()
      data = resp_data['list']  # get the city_id and cod

      if isinstance(data, list):
      # give the data to kafka topic to process
        stream_to_kafka(data, hist_we_prod, 'raw_weather_data')
      # insert data to clickhouse for storage
        insert_to_clickhouse(hist_we_cons, 'historical_weather_data', column_names, defaults)

        all_weather_data.extend(data)
      else:
        all_weather_data.append(data)
    except Exception as e:
      print(f"An error occurred: {e}")

    current_date = segment_end

print(f"Data collection and DB storage complete. Total records collected: {len(all_weather_data)}")
# print(json.dumps(all_weather_data, indent=2)) # set to print a few instead

In [ ]:
weather_list = json.dumps(all_weather_data, indent=2)

# print(json.dumps(all_weather_data[0], indent=2))

# 3. Parallelize the JSON string into a weather collection RDD
weather_collection = spark.sparkContext.parallelize([weather_list])

# 4. Create an initial DataFrame to inspect schema
raw_weather_df = spark.read.json(weather_collection)
schema_fields = raw_weather_df.columns
raw_weather_df = raw_weather_df.withColumns(
  {"weather": F.array_join(F.col("weather.main"), ", "),
   "weather_desc": F.array_join(F.col("weather.description"), ", "),
   "dt": F.from_unixtime(F.col("dt")),
  }
)

# raw_weather_df.show(n=3, truncate=False)
# raw_weather_df.printSchema()

# 5. Define selection expressions with safety checks for 'rain' and 'snow'
select_exprs = [
    F.col("dt").alias("date_obs"),
    F.col("main.temp").alias("Current_Temp  (\N{DEGREE SIGN}C)"),
    F.col("main.feels_like").alias("Feels_Like  (\N{DEGREE SIGN}C)"),
    F.col("main.pressure").alias("Pressure (hPa)"),
    F.col("main.humidity").alias("Humidity (%)"),
    F.col("main.temp_min").alias("Min_Temp (\N{DEGREE SIGN}C)"),
    F.col("main.temp_max").alias("Max_Temp (\N{DEGREE SIGN}C)"),
    F.col("wind.speed").alias("Wind_Speed (m/s)"),
    F.col("wind.deg").alias("Wind_Deg"),
    F.col("clouds.all").alias("Cloudiness (%)"),
    # Check for rain
    F.col("rain.1h") if "rain" in schema_fields else F.lit(None).alias("Rainfall (mm)"),
    # Check for snow
    F.col("snow.1h") if "snow" in schema_fields else F.lit(None).alias("Snowfall (mm)"),
    F.col("weather").alias("Weather_Main"), # weather.main is things like rain, clouds, snow etc.
    F.col("Weather_Desc"), # details on weather like cloud formation etc.
]

raw_weather_df = raw_weather_df.select(*select_exprs).sort(F.asc("dt"))
raw_weather_df.show(n=1, truncate=False)

In [ ]:
# daily statistical weather data (predicted) for SG
# to use it for our test/ validation dataset

# coordinates and information
lat = 1.290
long = 103.852
weather_api_key = userdata.get('WEATHER_API_KEY')

weather_pred_url = "https://history.openweathermap.org/data/2.5/aggregated/day"

# Define date range
current_date = datetime.datetime(2026, 3, 1)
end_date = datetime.datetime(2026, 4, 1) # to replace with dynamic mth later on
pred_weather_list = []

# general logic follows the same as above (refactor later)

print(f"Batch data prediction for period Mar-2026 \n")

# 2. Batch call API for predicted weather data
while current_date < end_date:
    daily_request_url = (
        f'{weather_pred_url}?lat={lat}&lon={long}'
        f'&month={current_date.month}&day={current_date.day}'
    )

    try:
      print(f"Fetching data for {current_date.date()}")
      pred_weather_response = requests.get(daily_request_url)

      resp_data = pred_weather_response.json()
      pred_weather_list.append(resp_data)
    except Exception as e:
      print(f"An error occurred: {e}")

    current_date += datetime.timedelta(days=1)

print(f"Data collection complete. Total records batches collected: {len(pred_weather_list)}")
# print(json.dumps(pred_weather_list, indent=2)) # set to print a few instead